# Q-MADDPG Phase 2 — 1.5M Steps — Modal A100 v3 FAST2

**Two optimisations to complete in ~16 hours (8h safety margin):**

1. `UPDATE_EVERY: 2 → 8` — 4× fewer gradient calls (~1.78× speedup)
2. **NumPy replay buffer** replacing Python deque — O(1) random access vs O(n)
   (~1.20× speedup on buf.sample() which runs millions of times)

**Combined: ~2.1× faster → estimated ~16 hours on A100**
**Safety margin: ~8 hours before Modal's 24h timeout**

**App:** `qmaddpg-1p5m-fast2` | **Volume:** `qmaddpg-outputs/QMADDPG_1p5M_v3_fast2`
**BATCH_SIZE=512 kept** — same training quality as original

In [10]:
# CELL 1 — Install
!pip install -q modal pettingzoo==1.24.3 gymnasium==0.29.1 supersuit==3.9.3 torch numpy matplotlib
print('Install complete')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.9/156.9 kB 4.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 47.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 979.4/979.4 kB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.4/460.4 kB 29.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.1 which is incompatible.
Install complete


In [11]:
# CELL 2 — Authenticate Modal
import os, subprocess, sys
os.environ["MODAL_TOKEN_ID"]     = "ak-aGXDeq3wM10Vf22peClYU8"
os.environ["MODAL_TOKEN_SECRET"] = "as-vfktrwarFbeyZPYTRoADoG"
config_dir = os.path.expanduser("~/.modal")
os.makedirs(config_dir, exist_ok=True)
with open(f"{config_dir}/config.toml", "w") as f:
    f.write(f'[default]\ntoken_id = "{os.environ["MODAL_TOKEN_ID"]}"\n'
            f'token_secret = "{os.environ["MODAL_TOKEN_SECRET"]}"\n')
r = subprocess.run([sys.executable, "-m", "modal", "token", "set",
                    "--token-id", os.environ["MODAL_TOKEN_ID"],
                    "--token-secret", os.environ["MODAL_TOKEN_SECRET"]],
                   capture_output=True, text=True)
print("✅ Authenticated" if r.returncode == 0 else f"❌ {r.stderr}")

✅ Authenticated


In [ ]:
# CELL 3 — Write FAST2 training script
import base64, os
script_b64 = "..."
script = base64.b64decode(script_b64).decode()
with open('/content/qmaddpg_1p5m_modal_fast2.py', 'w') as f:
    f.write(script)
size = os.path.getsize('/content/qmaddpg_1p5m_modal_fast2.py')
print(f"Script written — {size:,} bytes")
print("Optimisations: UPDATE_EVERY=8 + numpy replay buffer")
print("Estimated: ~16 hours on A100 (8h safety margin)")

Script written — 25,709 bytes
Optimisations: UPDATE_EVERY=8 + numpy replay buffer
Estimated: ~16 hours on A100 (8h safety margin)


In [ ]:
# CELL 4 — Verify syntax
import subprocess, sys, os
r = subprocess.run([sys.executable, '-m', 'py_compile', '/content/qmaddpg_1p5m_modal_fast2.py'],
                   capture_output=True, text=True)
print("✅ Syntax OK" if r.returncode == 0 else f"❌ {r.stderr}")

✅ Syntax OK


In [ ]:
# CELL 5 — SUBMIT
import subprocess, sys, os
print("Q-MADDPG 1.5M FAST2 — submitting to Modal A100...")
print("Estimated: ~16h | Safety margin: ~8h")
print("=" * 70)
result = subprocess.run(
    [sys.executable, "-m", "modal", "run", "--detach",
     "/content/qmaddpg_1p5m_modal_fast2.py"],
    capture_output=True, text=True, env={**os.environ}
)
print(result.stdout)
if result.stderr:
    lines = [l for l in result.stderr.split("\n")
             if "remote()" not in l and "map()" not in l]
    filtered = "\n".join(lines).strip()
    if filtered: print("STDERR:", filtered)
print(f"Return code: {result.returncode}")
if result.returncode == 0:
    print("\n✅ Running on Modal A100 — close Colab safely")
    print("   Done when: final_actor_0.pth in QMADDPG_1p5M_v3_fast2")
else:
    print("\n❌ Failed — re-run Cell 2 then retry")

Q-MADDPG 1.5M FAST2 — submitting to Modal A100...
Estimated: ~16h | Safety margin: ~8h
Note that running a local entrypoint in detached mode only keeps the last triggered Modal function alive after the parent process has been killed or disconnected.
✓ Initialized. View run at 
https://modal.com/apps/madhukumara2208/main/ap-nmzB2OkfRQfp7jQ121TnTR
Building image im-Y96uLvBXvI4KYqsAfb9tt5

=> Step 0: FROM base

=> Step 1: RUN python -m pip install gymnasium==0.29.1 matplotlib numpy pettingzoo==1.24.3 pygame supersuit==3.9.3 torch
Looking in indexes: http://pypi-mirror.modal.local:5555/simple
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 847.8/847.8 kB 200.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 498.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 122.7 MB/s eta 0:00:00
   ━

In [ ]:
# CELL 6 — Check progress
import subprocess, sys, os
r = subprocess.run([sys.executable, "-m", "modal", "app", "list"],
                   capture_output=True, text=True, env={**os.environ})
print(r.stdout if r.stdout else r.stderr)
r2 = subprocess.run([sys.executable, "-m", "modal", "volume", "ls",
                     "qmaddpg-outputs", "QMADDPG_1p5M_v3_fast2"],
                    capture_output=True, text=True, env={**os.environ})
print(r2.stdout if r2.stdout else "(still running)")
print("Done when: final_actor_0.pth appears")

                                            Apps                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━┳
┃ App ID                    ┃ Description ┃ State        ┃ Tasks ┃ Created at  ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━╇
│ ap-nmzB2OkfRQfp7jQ121TnTR │ qmaddpg-1p… │ ephemeral    │ 1     │ 2026-08-27  │
│                           │             │ (detached)   │       │ 19:46 UTC   │
└───────────────────────────┴─────────────┴──────────────┴───────┴─────────────┴

(still running)
Done when: final_actor_0.pth appears


In [7]:
# CELL 7 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_SAVE = '/content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_1p5M_v3_fast2'
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f"Drive ready: {DRIVE_SAVE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive ready: /content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_1p5M_v3_fast2


In [8]:
# CELL 8 — Download to Drive (after training complete)
import subprocess, sys, os
DRIVE_SAVE = '/content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_1p5M_v3_fast2'
os.makedirs(DRIVE_SAVE, exist_ok=True)
r = subprocess.run([sys.executable, "-m", "modal", "volume", "get",
                    "qmaddpg-outputs", "QMADDPG_1p5M_v3_fast2", f"{DRIVE_SAVE}/"],
                   capture_output=True, text=True, env={**os.environ})
print(r.stdout if r.stdout else r.stderr)
print("\nFiles saved:")
for f in sorted(os.listdir(DRIVE_SAVE)):
    full = os.path.join(DRIVE_SAVE, f)
    if os.path.isfile(full):
        print(f"  {f:50s} {os.path.getsize(full)/1024:7.1f} KB")

/usr/bin/python3: No module named modal


Files saved:


In [12]:
import subprocess, sys, os

from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE = '/content/drive/MyDrive/MTech_MARL_Phase2/QMADDPG_1p5M_v3_fast2'
LOCAL_TMP  = '/content/qmaddpg_download'
os.makedirs(DRIVE_SAVE, exist_ok=True)
os.makedirs(LOCAL_TMP,  exist_ok=True)

# Step 1: Download everything to local first
print("Step 1: Downloading from Modal volume to local...")
r = subprocess.run(
    [sys.executable, "-m", "modal", "volume", "get",
     "qmaddpg-outputs", "QMADDPG_1p5M_v3_fast2", f"{LOCAL_TMP}/"],
    capture_output=True, text=True, env={**os.environ}
)
print(r.stdout if r.stdout else r.stderr)

# Step 2: Copy ALL files flat to Drive
import shutil
print("\nStep 2: Copying all files to Drive...")
copied = []
for root, dirs, files in os.walk(LOCAL_TMP):
    for fname in files:
        src  = os.path.join(root, fname)
        rel  = os.path.relpath(src, LOCAL_TMP)
        flat = rel.replace('/', '_').replace('\\', '_')
        dst  = os.path.join(DRIVE_SAVE, flat)
        shutil.copy(src, dst)
        size = os.path.getsize(dst) / 1024
        print(f"  {flat:60s} {size:8.1f} KB")
        copied.append(flat)

print(f"\n✅ {len(copied)} files saved to Drive: {DRIVE_SAVE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Step 1: Downloading from Modal volume to local...
✓ Finished downloading files to local!


Step 2: Copying all files to Drive...
  QMADDPG_1p5M_v3_fast2_qmaddpg_v3_1p5m_log.csv                  2071.8 KB
  QMADDPG_1p5M_v3_fast2_best_critic_1.pth                         336.2 KB
  QMADDPG_1p5M_v3_fast2_best_actor_0.pth                           30.4 KB
  QMADDPG_1p5M_v3_fast2_final_actor_0.pth                          30.4 KB
  QMADDPG_1p5M_v3_fast2_best_actor_2.pth                           30.4 KB
  QMADDPG_1p5M_v3_fast2_qmaddpg_v3_1p5M_curves.png                189.9 KB
  QMADDPG_1p5M_v3_fast2_final_critic_0.pth                        336.2 KB
  QMADDPG_1p5M_v3_fast2_best_critic_2.pth                         336.2 KB
  QMADDPG_1p5M_v3_fast2_final_actor_2.pth                          30.4 KB
  QMADDPG_1p5M_v3_fast2_best_actor_1.pth                           